In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [3]:
sns.set_style('whitegrid')

In [4]:
df = pd.read_csv('heart_disease_dataset.csv')
print("--- Initial Info ---")
df.info()

--- Initial Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Age                      1000 non-null   int64 
 1   Gender                   1000 non-null   object
 2   Cholesterol              1000 non-null   int64 
 3   Blood Pressure           1000 non-null   int64 
 4   Heart Rate               1000 non-null   int64 
 5   Smoking                  1000 non-null   object
 6   Alcohol Intake           660 non-null    object
 7   Exercise Hours           1000 non-null   int64 
 8   Family History           1000 non-null   object
 9   Diabetes                 1000 non-null   object
 10  Obesity                  1000 non-null   object
 11  Stress Level             1000 non-null   int64 
 12  Blood Sugar              1000 non-null   int64 
 13  Exercise Induced Angina  1000 non-null   object
 14  Chest Pain Type     

In [5]:
print('\nBefore replacing None or NaN values in Alcohol Intake:')
print(df['Alcohol Intake'].unique())

# মিসিং ভ্যালু 'Never' দিয়ে পূরণ করা
df['Alcohol Intake'] = df['Alcohol Intake'].fillna('Never')

print('After replacing None or NaN values:')
print(df['Alcohol Intake'].unique())


Before replacing None or NaN values in Alcohol Intake:
['Heavy' nan 'Moderate']
After replacing None or NaN values:
['Heavy' 'Never' 'Moderate']


In [6]:
le = LabelEncoder()
categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = df.copy()

print('\nEncoding mapping :')
print('-' * 30)
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'{col} : {mapping}')


Encoding mapping :
------------------------------
Gender : {'Female': np.int64(0), 'Male': np.int64(1)}
Smoking : {'Current': np.int64(0), 'Former': np.int64(1), 'Never': np.int64(2)}
Alcohol Intake : {'Heavy': np.int64(0), 'Moderate': np.int64(1), 'Never': np.int64(2)}
Family History : {'No': np.int64(0), 'Yes': np.int64(1)}
Diabetes : {'No': np.int64(0), 'Yes': np.int64(1)}
Obesity : {'No': np.int64(0), 'Yes': np.int64(1)}
Exercise Induced Angina : {'No': np.int64(0), 'Yes': np.int64(1)}
Chest Pain Type : {'Asymptomatic': np.int64(0), 'Atypical Angina': np.int64(1), 'Non-anginal Pain': np.int64(2), 'Typical Angina': np.int64(3)}


In [7]:
numerical_cols = ['Age', 'Cholesterol', 'Blood Pressure', 'Heart Rate', 'Stress Level', 'Blood Sugar']
scaler = StandardScaler()

df_scaled = df_encoded.copy()
df_scaled[numerical_cols] = scaler.fit_transform(df_scaled[numerical_cols])

# ডেপ্লয়মেন্টের সময় নতুন ডেটা স্কেল করার জন্য এই scaler-টি সেভ করা জরুরি
joblib.dump(scaler, "scaler.pkl")
print("\n[INFO] Scaler saved successfully as 'scaler.pkl'")


[INFO] Scaler saved successfully as 'scaler.pkl'


In [8]:
X = df_scaled.drop('Heart Disease', axis=1)
y = df_scaled['Heart Disease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf'),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
    "KNN (K=5)": KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree', metric='euclidean')
}

# বেস্ট মডেল ট্র্যাক করার জন্য ভ্যারিয়েবল
best_accuracy = 0
best_model_name = ""
best_model = None

In [10]:
print("\n--- Training and Evaluating Models ---")
for name, model in models.items():
    # মডেল ট্রেইন করা
    model.fit(X_train, y_train)
    # অ্যাকুরিসি চেক করা
    acc = model.score(X_test, y_test)
    print(f"{name} -> Accuracy: {acc:.4f}")
    
    # যদি এই মডেলের অ্যাকুরিসি আগের চেয়ে ভালো হয়, তবে এটিকে বেস্ট হিসেবে সেট করা
    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name
        best_model = model

print("\n" + "="*40)
print(f"🏆 BEST MODEL FOUND: {best_model_name}")
print(f"📈 Accuracy: {best_accuracy:.4f}")
print("="*40)

# বেস্ট মডেলটির ডিটেইলড রিপোর্ট দেখা
y_pred = best_model.predict(X_test)
print("\n--- Best Model Classification Report ---")
print(classification_report(y_test, y_pred))


--- Training and Evaluating Models ---
Logistic Regression -> Accuracy: 0.8700
Decision Tree -> Accuracy: 1.0000
Random Forest -> Accuracy: 0.9900
SVM -> Accuracy: 0.8900
Gradient Boosting -> Accuracy: 1.0000
Neural Network -> Accuracy: 0.9550
KNN (K=5) -> Accuracy: 0.8800

🏆 BEST MODEL FOUND: Decision Tree
📈 Accuracy: 1.0000

--- Best Model Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       122
           1       1.00      1.00      1.00        78

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



c:\Users\hasha\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [11]:
joblib.dump(best_model, "heart_model.pkl")
print(f"\n[SUCCESS] Successfully saved {best_model_name} as 'heart_model.pkl'!")


[SUCCESS] Successfully saved Decision Tree as 'heart_model.pkl'!
